<a href="https://colab.research.google.com/github/M-Santhoshkumar-2005/ITA0616-MACHINE-LEARNING-SLOT-A-/blob/main/Experiment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from collections import Counter
import numpy as np
import pandas as pd


# =====================================================================
# Embedded Dataset (Based on Unit IV Maths & CS Pass/Fail Concept Map)
# Columns: [Maths_Score, CS_Score, Result]  ('P' = Pass, 'F' = Fail)
# =====================================================================
data = [
    [4, 3, "F"],
    [6, 7, "P"],
    [7, 8, "P"],
    [5, 5, "F"],
    [8, 8, "P"],
    [3, 4, "F"],
    [6, 6, "P"],
    [4, 5, "F"],
    [7, 7, "P"],
    [5, 4, "F"],
    [8, 7, "P"],
    [3, 2, "F"],
    [6, 8, "P"],
    [5, 6, "P"],
    [4, 4, "F"],
    [9, 8, "P"],
]

df = pd.DataFrame(data, columns=["Maths", "CS", "Result"])
print("Dataset Preview:")
print(df.to_string(index=False))
print()


def euclidean_distance(p1, p2):
    return np.sqrt(np.sum((np.array(p1, dtype=float) - np.array(p2, dtype=float)) ** 2))


def knn_predict_single(X_train, y_train, query_point, k=3):
    distances = []
    for idx, (x_row, label) in enumerate(zip(X_train, y_train)):
        dist = euclidean_distance(query_point, x_row)
        distances.append((dist, idx, x_row, label))

    # Sort by Euclidean distance ascending
    distances.sort(key=lambda item: item[0])
    k_nearest = distances[:k]

    # Majority vote among k nearest neighbours
    votes = Counter(item[3] for item in k_nearest)
    predicted_label = votes.most_common(1)[0][0]
    return predicted_label, k_nearest, distances


def knn_predict(X_train, y_train, X_test, k=3):
    return [knn_predict_single(X_train, y_train, x_q, k=k)[0] for x_q in X_test]


X_all = df[["Maths", "CS"]].values
y_all = df["Result"].values

# --- Part 1: Query Example (Maths=6, CS=8, k=3) ---
query = [6, 8]
k_val = 3
pred, nearest, all_dists = knn_predict_single(X_all, y_all, query, k=k_val)

print(f"=== Query Point: Maths={query[0]}, CS={query[1]} (k={k_val}) ===")
print("Top K Nearest Neighbours:")
for rank, (dist, idx, row, lbl) in enumerate(nearest, start=1):
    print(f"  {rank}. Row {idx + 1:2d} -> Maths={row[0]}, CS={row[1]}, Result={lbl} | Distance={dist:.4f}")
print(f"Predicted Result: {pred}\n")

# --- Part 2: Train / Test Evaluation ---
split_idx = int(0.75 * len(X_all))
X_train, y_train = X_all[:split_idx], y_all[:split_idx]
X_test, y_test = X_all[split_idx:], y_all[split_idx:]

for k_test in [1, 3, 5]:
    test_preds = knn_predict(X_train, y_train, X_test, k=k_test)
    accuracy = np.mean(np.array(test_preds) == y_test) * 100
    print(f"k={k_test} | Test Actual: {list(y_test)} | Test Predicted: {test_preds} | Accuracy: {accuracy:.2f}%")

Dataset Preview:
 Maths  CS Result
     4   3      F
     6   7      P
     7   8      P
     5   5      F
     8   8      P
     3   4      F
     6   6      P
     4   5      F
     7   7      P
     5   4      F
     8   7      P
     3   2      F
     6   8      P
     5   6      P
     4   4      F
     9   8      P

=== Query Point: Maths=6, CS=8 (k=3) ===
Top K Nearest Neighbours:
  1. Row 13 -> Maths=6, CS=8, Result=P | Distance=0.0000
  2. Row  2 -> Maths=6, CS=7, Result=P | Distance=1.0000
  3. Row  3 -> Maths=7, CS=8, Result=P | Distance=1.0000
Predicted Result: P

k=1 | Test Actual: ['P', 'P', 'F', 'P'] | Test Predicted: ['P', 'F', 'F', 'P'] | Accuracy: 75.00%
k=3 | Test Actual: ['P', 'P', 'F', 'P'] | Test Predicted: ['P', 'P', 'F', 'P'] | Accuracy: 100.00%
k=5 | Test Actual: ['P', 'P', 'F', 'P'] | Test Predicted: ['P', 'F', 'F', 'P'] | Accuracy: 75.00%
